# מפגש 10 — למה Accuracy גבוה עלול להטעות?
**יחידה 3 | מאגרים לא מאוזנים | הדגמה מודרכת על Stroke Prediction**

המטרה: להבין חוסר איזון בסיווג, לחשב יחס מחלקות, להשוות Accuracy/Recall/F1 ולבחור מדד שמתאים לשאלת חקר. **הדוגמה לימודית בלבד ואינה מודל רפואי.**

## 1. בעיית החקר
במקרים נדירים המחלקה החיובית קטנה. מה יקרה אם מודל תמיד ינבא 'לא אירע שבץ'? תחילה נספור דוגמאות.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, confusion_matrix, classification_report

url="https://raw.githubusercontent.com/arielb30/datasets/main/healthcare-dataset-stroke-data.csv"
df=pd.read_csv(url)
print(df.shape)
display(df.head())
print(df['stroke'].value_counts())

## 2. הבסיס: תמיד בוחרים את המחלקה הנפוצה
זהו מסווג Dummy שאינו לומד קשרים בין מאפייני הקלט למטרה. הוא עוזר להבין למה Accuracy עלול להטעות.

In [ ]:
features=["age","avg_glucose_level","bmi"]
X=df[features]
y=df["stroke"]
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,stratify=y,random_state=42)
base=DummyClassifier(strategy="most_frequent")
base.fit(X_train,y_train)
base_pred=base.predict(X_test)
print("Accuracy:",round(accuracy_score(y_test,base_pred),3))
print("Recall for stroke:",round(recall_score(y_test,base_pred,zero_division=0),3))
print("F1:",round(f1_score(y_test,base_pred,zero_division=0),3))

## 3. KNN — מודל אמיתי
נבחר שלושה מאפיינים מספריים כדי לא להעמיס קידוד קטגוריות בשיעור. Pipeline ממלא ערכים חסרים ומתקנן **לפי Train בלבד**.

In [ ]:
pipe=make_pipeline(SimpleImputer(strategy="median"),StandardScaler(),KNeighborsClassifier(n_neighbors=5))
pipe.fit(X_train,y_train)
pred=pipe.predict(X_test)
print(classification_report(y_test,pred,zero_division=0))
print("Confusion matrix (row=true, col=pred):\n",confusion_matrix(y_test,pred,labels=[0,1]))

## 4. Accuracy מול F1 בבחירת K
לבחירת K באופן מקצועי נשתמש ב־Cross Validation **רק בנתוני Train**. זהו ניסוי המשך: אין לבחור K על סמך אותו Test ואז לדווח עליו כמבחן סופי.

In [ ]:
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
for metric in ["accuracy","f1"]:
    print("Scoring:",metric)
    for k in [3,5,9,15]:
        p=make_pipeline(SimpleImputer(strategy="median"),StandardScaler(),KNeighborsClassifier(n_neighbors=k))
        scores=cross_val_score(p,X_train,y_train,scoring=metric,cv=cv)
        print(k,round(scores.mean(),3))

## 5. דיון: מה מייצג F1 ומה אינו מייצג?
F1 משקלל Precision ו־Recall; לעיתים יעדיף מודל שונה מזה המוביל ב־Accuracy. מדד מתאים תלוי גם במחיר טעויות. לעולם אין לפרש את המדדים כאן ככלי לחיזוי רפואי אישי.

## 6. העשרה: Sampling ו־SMOTE
המצגת מציגה Undersampling, Oversampling ו־SMOTE. יש לבצע כל דגימה מחדש **בתוך קיפולי האימון**, אחרי פיצול Train/Test, באמצעות imbalanced-learn Pipeline. **אין** לאזן Test; אין להפעיל SMOTE על כל ה־Dataset לפני החלוקה.

## כרטיס יציאה
1. כיצד מודל שרוב תחזיותיו נכונות יכול להחמיץ כמעט את כל המקרים החיוביים?
2. מהו ההבדל בין Accuracy ל־Recall?
3. למה משתמשים ב־stratify=y?
4. מה משמעות ה־F1 במדגם לא מאוזן?
5. מה נדרש לתיעוד בפרויקט?